# Laboratory 1 — Leaky Integrate-and-Fire Neuron

Four worked examples accompany Chapter 3: current/rate, RC charging,
temporal integration, and multisource integration. **Run all cells once.
Then edit one parameter cell and rerun its example.** Code is self-contained
and uses NumPy and Matplotlib (available in Colab). All values are teaching
assumptions, not fitted biological measurements.

[Run in Google Colab](https://colab.research.google.com/github/an-hongyu/Introduction-to-Neuromorphic-Embodied-AI/blob/main/notebooks/lif_neuron.ipynb)

Units: pA, mV, pF, nS, ms. Stored voltage resets at spike detection;
short vertical markers indicate separate spike events, not ionic waveforms.
The dotted curves in integration plots show the passive RC prediction
with threshold and reset disabled.

## 1. Model and plotting helpers

A current array specifies one value per interval. The returned voltage
includes the initial state, so it is one sample longer. The exponential
step is exact for a constant current within an interval; threshold detection
is at interval endpoints. Refractory duration rounds upward to a full step.
Use smaller steps to check timing convergence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_lif(current_pa, dt_ms=0.1, capacitance_pf=200.0,
                 leak_ns=10.0, rest_mv=-65.0, threshold_mv=-50.0,
                 reset_mv=-65.0, refractory_ms=2.0, spiking=True):
    """One current value per time interval; return time, voltage, spike times.

    Units: pA, ms, pF, nS, mV. Initial voltage is reset_mv.
    Exact passive update for piecewise-constant current; endpoint events.
    spiking=False exposes the passive RC response without threshold/reset.
    """
    current = np.asarray(current_pa, dtype=float)
    if current.ndim != 1 or current.size == 0 or not np.isfinite(current).all():
        raise ValueError('Supply a nonempty, finite 1-D current array.')
    if dt_ms <= 0 or capacitance_pf <= 0 or leak_ns <= 0 or refractory_ms < 0:
        raise ValueError('dt, capacitance, and leak must be positive; refractory >= 0.')
    if reset_mv >= threshold_mv:
        raise ValueError('Reset must be below threshold.')
    tau_ms = capacitance_pf / leak_ns
    decay = np.exp(-dt_ms / tau_ms)
    time = np.arange(current.size + 1) * dt_ms
    voltage = np.empty(time.size)
    voltage[0] = reset_mv
    spikes, hold = [], 0
    # Round refractory duration upward to a whole number of time steps.
    refractory_steps = int(np.ceil(refractory_ms / dt_ms - 1e-12))
    for k, injected_pa in enumerate(current):
        if hold > 0:
            voltage[k + 1] = reset_mv
            hold -= 1
        else:
            steady_mv = rest_mv + injected_pa / leak_ns
            voltage[k + 1] = steady_mv + (voltage[k] - steady_mv) * decay
            if spiking and voltage[k + 1] >= threshold_mv:
                spikes.append(time[k + 1])
                voltage[k + 1] = reset_mv
                hold = refractory_steps
    return time, voltage, np.asarray(spikes)

def pulse_train(time_ms, starts_ms, width_ms=4.0, amplitude_pa=600.0):
    """Rectangular current pulses; time_ms contains interval start times."""
    current = np.zeros_like(time_ms, dtype=float)
    for start in starts_ms:
        current += amplitude_pa * ((time_ms >= start) & (time_ms < start + width_ms))
    return current

def analytical_rate(current_pa, capacitance_pf=200.0, leak_ns=10.0,
                    rest_mv=-65.0, threshold_mv=-50.0, reset_mv=-65.0,
                    refractory_ms=2.0):
    steady_mv = rest_mv + current_pa / leak_ns
    if steady_mv <= threshold_mv:
        return 0.0
    crossing_ms = (capacitance_pf / leak_ns) * np.log(
        (steady_mv - reset_mv) / (steady_mv - threshold_mv))
    return 1000.0 / (crossing_ms + refractory_ms)

def plot_response(ax, time, voltage, spikes, label, color, threshold=-50.0):
    ax.plot(time, voltage, color=color, label=label, lw=1.8)
    if len(spikes):
        ax.vlines(spikes, threshold + 0.7, threshold + 3.0, color=color, lw=1.5)
    ax.axhline(threshold, color='0.55', ls='--', lw=0.9)
    ax.set(xlabel='Time (ms)', ylabel='Voltage (mV)')

## 2. Example 1 — Change one neuron parameter

Predict first, then run. At the defaults, $R_m=100$ MΩ, $\tau_m=20$ ms,
first crossing is 27.73 ms, and the long-run rate is about 33.64 Hz.
Try current 100/200/300 pA; capacitance 100/200/400 pF; leak 5/10/20 nS;
threshold −55/−50/−45 mV; refractory 0/2/5 ms.
Change one value at a time. Capacitance changes timing without changing
the passive steady voltage; changing leak changes both.

In [ ]:
# EDIT these numbers, then rerun this cell and Example 1.
dt_ms = 0.1
duration_ms = 500.0
current_pa = 200.0
capacitance_pf = 200.0
leak_ns = 10.0
threshold_mv = -50.0
refractory_ms = 2.0
parameters = dict(dt_ms=dt_ms, capacitance_pf=capacitance_pf,
                  leak_ns=leak_ns, threshold_mv=threshold_mv,
                  refractory_ms=refractory_ms)
print(f'R = {1000 / leak_ns:.1f} MΩ; tau = {capacitance_pf / leak_ns:.1f} ms')

In [ ]:
current = np.full(round(duration_ms / dt_ms), current_pa)
time, voltage, spikes = simulate_lif(current, **parameters)
rate = 0.0 if len(spikes) < 2 else 1000.0 / np.diff(spikes).mean()
print(f'{len(spikes)} spikes in {duration_ms:g} ms; interval-based rate = {rate:.2f} Hz')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.1), constrained_layout=True)
plot_response(axes[0], time, voltage, spikes, 'Voltage / spike events', '#24628a', threshold_mv)
axes[0].set(xlim=(0, min(160, duration_ms)), title='A. Change the parameters above')
axes[0].legend(fontsize=8, loc='lower right')
currents = np.linspace(80, 300, 23)
simulated = []
for value in currents:
    _, _, events = simulate_lif(np.full(round(2000 / dt_ms), value), **parameters)
    simulated.append(0.0 if len(events) < 2 else 1000 / np.diff(events).mean())
expected = [analytical_rate(i, **{k: v for k, v in parameters.items() if k != 'dt_ms'})
            for i in currents]
axes[1].plot(currents, expected, color='#24628a', label='Analytical')
axes[1].plot(currents, simulated, 'o', ms=3.5, color='#b6483b', label='Simulated')
axes[1].set(xlabel='Current (pA)', ylabel='Steady firing rate (Hz)', title='B. Current–rate relation')
axes[1].legend(fontsize=9)
plt.show()

## 3. Example 2 — Charge the membrane capacitor

The 100 pA step has a passive equilibrium of −55 mV at 10 nS leak.
Different capacitances approach the **same** equilibrium at different rates.
At one time constant after onset, the voltage completes about 63.2% of its
final shift. No threshold/reset is applied in this RC-only example.
Edit the capacitance list and the pulse amplitude or width.

In [ ]:
# Compare capacitance while keeping resistance and current fixed.
dt_rc = 0.1
sample_rc = np.arange(1000) * dt_rc
step_current = pulse_train(sample_rc, [10], width_ms=80, amplitude_pa=100)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for cap, color in [(100, '#b6483b'), (200, '#24628a'), (400, '#268471')]:
    t, v, _ = simulate_lif(step_current, dt_ms=dt_rc, capacitance_pf=cap,
                          spiking=False)
    axes[0].plot(t, v, color=color, label=f'C = {cap} pF; tau = {cap / 10:g} ms')
axes[0].set(xlabel='Time (ms)', ylabel='Voltage (mV)', title='A. Larger C charges more slowly')
axes[0].legend(fontsize=9)
axes[1].step(sample_rc, step_current, where='post', color='#24628a')
axes[1].set(xlabel='Time (ms)', ylabel='Applied current (pA)', title='B. Same 100 pA input')
plt.show()

## 4. Example 3 — Temporal integration

The same input source sends two 600 pA, 4 ms pulses. At 6 ms onset
separation (2 ms quiet gap), their residual voltages combine and trigger
one event; at 40 ms separation, neither pulse triggers an event.
Pulse charge is unchanged. Only arrival timing differs.
Try separation 6/12/20/40 ms, then change capacitance.
If pulses overlap, their currents add: that is a different stimulus.

In [ ]:
# EDIT separation_ms (6 -> 40), pulse_pa, or capacitance_pf here.
separation_ms = 6.0
pulse_pa = 600.0
temporal_capacitance_pf = 200.0
sample = np.arange(1000) * 0.1
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True, constrained_layout=True)
for column, separation in enumerate([separation_ms, 40.0]):
    current = pulse_train(sample, [20, 20 + separation], width_ms=4,
                          amplitude_pa=pulse_pa)
    t, v, events = simulate_lif(current, capacitance_pf=temporal_capacitance_pf)
    _, passive, _ = simulate_lif(current, capacitance_pf=temporal_capacitance_pf,
                                  spiking=False)
    plot_response(axes[0, column], t, v, events, 'LIF + events', '#24628a')
    axes[0, column].plot(t, passive, ':', color='#b6483b', label='Passive RC, no reset')
    axes[0, column].set(title=f'Onset separation = {separation:g} ms; spikes = {len(events)}', ylim=(-66, -43))
    axes[0, column].legend(fontsize=8, loc='lower right')
    axes[1, column].step(sample, current, where='post', color='#24628a')
    axes[1, column].set(xlabel='Time (ms)', ylabel='Input current (pA)', xlim=(0, 100), ylim=(-20, 650))
plt.show()

## 5. Example 4 — Multiple sources: point-neuron spatial summation

A and B are distinct presynaptic input channels, each 250 pA for 8 ms.
One channel alone is subthreshold; their synchronous sum triggers one
event. Adding −150 pA during the pulse prevents that event at defaults.
Change B's weight or delay and observe coincidence sensitivity.

This is **multisource summation into one isopotential compartment**.
It does not model synaptic positions, dendritic distance, or cable filtering.
Negative current is a current-based inhibitory approximation; it does not
implement conductance-dependent shunting or an inhibitory reversal potential.
The weights are fixed inputs to this experiment, not learned synaptic weights.

In [ ]:
# Separate rows are different input sources, all reaching one compartment.
# EDIT weight_b, delay_b_ms, or inhibitory_pa; the other examples stay fixed.
weight_b = 1.0
delay_b_ms = 0.0
inhibitory_pa = -150.0
sample = np.arange(800) * 0.1
source_a = pulse_train(sample, [30], width_ms=8, amplitude_pa=250)
source_b = pulse_train(sample, [30 + delay_b_ms], width_ms=8, amplitude_pa=250)
source_i = pulse_train(sample, [30], width_ms=8, amplitude_pa=inhibitory_pa)
channels = np.vstack([source_a, source_b, source_i])
cases = [('A only', np.array([1., 0., 0.])),
         ('A + B', np.array([1., weight_b, 0.])),
         ('A + B + negative current', np.array([1., weight_b, 1.]))]
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, constrained_layout=True)
for col, (label, weights) in enumerate(cases):
    current = weights @ channels
    t, v, events = simulate_lif(current)
    _, passive, _ = simulate_lif(current, spiking=False)
    plot_response(axes[0, col], t, v, events, 'LIF + events', '#24628a')
    axes[0, col].plot(t, passive, ':', color='#b6483b', label='Passive RC')
    axes[0, col].set(title=f'{label}\nspikes = {len(events)}', ylim=(-66, -44))
    axes[0, col].legend(fontsize=8, loc='lower right')
    axes[1, col].step(sample, current, where='post', color='#24628a')
    axes[1, col].set(xlabel='Time (ms)', ylabel='Total current (pA)', xlim=(15, 65), ylim=(-20, 550))
plt.show()

## 6. Reference checks

These checks always use the documented baseline values, independently
of your edits. They test numerical behavior, not biological validity.

In [ ]:
# Reference checks use explicit defaults, independent of your editable cells.
_, zero_v, zero_events = simulate_lif(np.zeros(1000))
assert np.allclose(zero_v, -65) and len(zero_events) == 0
_, _, low = simulate_lif(np.full(5000, 100.0))
assert len(low) == 0
crossing = 20 * np.log(4)
_, _, coarse = simulate_lif(np.full(5000, 200.0))
_, _, fine = simulate_lif(np.full(10000, 200.0), dt_ms=0.05)
assert abs(coarse[0] - crossing) <= 0.1
assert abs(fine[0] - crossing) <= abs(coarse[0] - crossing) + 1e-10
assert abs(np.diff(fine).mean() - (crossing + 2)) <= 0.05
_, _, slower = simulate_lif(np.full(10000, 200.0), capacitance_pf=400)
assert abs(slower[0] - 2 * crossing) <= 0.1
rate_errors = []
for drive in np.linspace(80, 300, 23):
    _, _, events = simulate_lif(np.full(20000, drive))
    measured = 0.0 if len(events) < 2 else 1000 / np.diff(events).mean()
    rate_errors.append(abs(measured - analytical_rate(drive)))
assert max(rate_errors) < 0.6
# A passive step should agree with the analytical RC charging curve.
t, v, _ = simulate_lif(np.full(1000, 100.0), spiking=False)
assert np.allclose(v, -55 + (-65 + 55) * np.exp(-t / 20))
sample = np.arange(1000) * 0.1
assert len(simulate_lif(pulse_train(sample, [20, 26]))[2]) == 1
assert len(simulate_lif(pulse_train(sample, [20, 60]))[2]) == 0
a = pulse_train(sample, [30], width_ms=8, amplitude_pa=250)
negative = pulse_train(sample, [30], width_ms=8, amplitude_pa=-150)
assert len(simulate_lif(a)[2]) == 0
assert len(simulate_lif(2 * a)[2]) == 1
assert len(simulate_lif(2 * a + negative)[2]) == 0
# Superposition applies to passive voltage shifts, before threshold/reset.
_, va, _ = simulate_lif(a, spiking=False)
_, vab, _ = simulate_lif(2 * a, spiking=False)
assert np.allclose(vab + 65, 2 * (va + 65))
print('Passed: baseline, analytical RC, spike timing, step refinement, temporal and multisource integration.')
print(f'Default first spike: {coarse[0]:.2f} ms; analytical: {crossing:.5f} ms')
print(f'Default interval rate: {1000 / np.diff(coarse).mean():.2f} Hz; analytical: {analytical_rate(200):.2f} Hz')

## Explain what changed

For each example, record the parameter value and unit, a prediction,
the voltage/input plot, spike count or first-spike time, and an explanation
in terms of charge, leak, time constant, and threshold. These are guided
worked examples; the chapter exercises ask you to transfer the ideas.

## References

[1] W. Gerstner, W. M. Kistler, R. Naud, and L. Paninski,
*Neuronal Dynamics: From Single Neurons to Networks and Models of Cognition*.
Cambridge, U.K.: Cambridge University Press, 2014, secs. 1.3–1.4.
[RC circuit and integration](https://neuronaldynamics.epfl.ch/online/Ch1.S3.html);
[limitations](https://neuronaldynamics.epfl.ch/online/Ch1.S4.html).
Circuit and plots in this textbook are original teaching illustrations.